In [3]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import lightgbm as lgb

In [4]:
df = pd.read_csv("../data/week1_features.csv")
df.columns = df.columns.str.replace('[', '_', regex=False).str.replace(']', '_', regex=False).str.replace(' ', '_', regex=False)

feature_cols = [col for col in df.columns if 'roll' in col]
X = df[feature_cols]
y = df["Machine_failure"]

print("Before SMOTE:")
print(y.value_counts())

Before SMOTE:
Machine_failure
0    9661
1     339
Name: count, dtype: int64


In [5]:
# SMOTE only inside CV folds
skf = StratifiedKFold(n_splits=5)
smote = SMOTE(random_state=42)
f1_scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # SMOTE only on training data!
    X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
    
    print(f"\nFold {fold+1}:")
    print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
    print(f"After SMOTE: {pd.Series(y_resampled).value_counts().to_dict()}")
    
    # Train model
    model = lgb.LGBMClassifier(n_estimators=100, random_state=42)
    model.fit(X_resampled, y_resampled)
    
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    f1_scores.append(f1)
    print(f"Macro F1: {f1:.4f}")

print(f"\nAverage Macro F1: {sum(f1_scores)/len(f1_scores):.4f}")


Fold 1:
Before SMOTE: {0: 7728, 1: 272}
After SMOTE: {0: 7728, 1: 7728}
[LightGBM] [Info] Number of positive: 7728, number of negative: 7728
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009803 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12739
[LightGBM] [Info] Number of data points in the train set: 15456, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Macro F1: 0.5060

Fold 2:
Before SMOTE: {0: 7729, 1: 271}
After SMOTE: {0: 7729, 1: 7729}
[LightGBM] [Info] Number of positive: 7729, number of negative: 7729
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004358 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12740
[LightGBM] [Info] Number of data points in the train set: 15458, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore